# Big Data Foundation – Final Day
# 🏭 IoT Sensor Analytics with PySpark

**Goal:** Build an end-to-end Big Data pipeline: IoT Sensors → Spark → Processing → Anomaly Detection → Business Insight.

Concepts: Big Data 5Vs, PySpark DataFrames, filtering, transformation, aggregation, anomaly detection, batch vs streaming, real-world architecture.


In [ ]:
!pip install -q pyspark


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark=(SparkSession.builder.appName("IoT_BigData_Analytics").master("local[*]").getOrCreate())
print("Spark Version:",spark.version)


## 1. Problem
A smart factory has machines continuously producing temperature, pressure and vibration readings. The business question is: **Which machines may need attention?**


In [ ]:
import random
from datetime import datetime,timedelta

machines=["Machine_A","Machine_B","Machine_C","Machine_D","Machine_E"]
locations=["Factory_1","Factory_2","Factory_3"]
data=[]
start_time=datetime(2026,8,20,9,0,0)

for i in range(100000):
    machine=random.choice(machines)
    location=random.choice(locations)
    timestamp=start_time+timedelta(seconds=i)
    data.append((i+1,timestamp.strftime("%Y-%m-%d %H:%M:%S"),machine,location,
                 round(random.uniform(60,95),2),round(random.uniform(20,50),2),
                 round(random.uniform(1,10),2)))

columns=["Sensor_ID","Timestamp","Machine","Location","Temperature","Pressure","Vibration"]
sensor_df=spark.createDataFrame(data,columns)
print("Total records:",sensor_df.count())


## 2. Explore the structured sensor data
Each reading has a defined schema: ID, timestamp, machine, location, temperature, pressure and vibration.


In [ ]:
sensor_df.show(10,truncate=False)
sensor_df.printSchema()


## 3. Filter abnormal temperatures
Business rule: temperature above 90°C is a potential warning.


In [ ]:
high_temperature=sensor_df.filter(col("Temperature")>90)
print("High-temperature readings:",high_temperature.count())
high_temperature.show(20,truncate=False)


## 4. Machine-wise aggregation
Question: **Which machine has the highest average temperature?** This demonstrates GroupBy + Aggregation.


In [ ]:
machine_summary=(sensor_df.groupBy("Machine").agg(
    avg("Temperature").alias("Avg_Temperature"),
    max("Temperature").alias("Max_Temperature"),
    avg("Pressure").alias("Avg_Pressure"),
    avg("Vibration").alias("Avg_Vibration")
).orderBy(desc("Avg_Temperature")))
machine_summary.show()


## 5. Rule-based anomaly detection
`Temperature > 90 OR Vibration > 8` → **WARNING**; otherwise → **NORMAL**.


In [ ]:
health_df=sensor_df.withColumn(
    "Machine_Status",
    when((col("Temperature")>90)|(col("Vibration")>8),"WARNING").otherwise("NORMAL")
)
health_df.select("Machine","Temperature","Vibration","Machine_Status").show(20)


## 6. Find the most problematic machine


In [ ]:
warning_summary=(health_df.filter(col("Machine_Status")=="WARNING")
    .groupBy("Machine").count()
    .withColumnRenamed("count","Warning_Count")
    .orderBy(desc("Warning_Count")))
warning_summary.show()


## 7. Visualize machine warnings


In [ ]:
import matplotlib.pyplot as plt
warning_pd=warning_summary.toPandas()
plt.figure(figsize=(8,5))
plt.bar(warning_pd["Machine"],warning_pd["Warning_Count"])
plt.title("Machine Warning Events")
plt.xlabel("Machine"); plt.ylabel("Number of Warnings")
plt.tight_layout(); plt.show()


## 8. Business insight
The pipeline turns 100,000 raw readings into machine-level warnings. This is the key Big Data idea: **large-scale data → processing → useful decision**.


## 9. Batch vs Streaming
**Batch:** Sensors → Storage → Spark → Report

**Streaming:** Sensor → Kafka/Event System → Spark Structured Streaming → Real-time Detection → Alert

Example: 65°C NORMAL → 75°C NORMAL → 94°C WARNING → 98°C CRITICAL.


## 10. Production architecture
```text
IoT Sensors
    ↓
Kafka / Event Platform
    ↓
Spark Structured Streaming
    ↓
Cleaning + Transformation + Analytics
    ↓
Anomaly Detection
    ↓
Dashboard / Alert
    ↓
Business Decision
```

Applications: manufacturing, banking fraud detection, healthcare monitoring, cybersecurity, e-commerce and fleet monitoring.


## 🎯 Final Student Challenge
1. Which machine has the highest maximum temperature?
2. Which machine has the highest average vibration?
3. How many WARNING events exist?
4. Which location has the most warnings?
5. Change the rule to Temperature > 92 or Vibration > 8.5.
6. Create a chart of warnings by location.


In [ ]:
print("Highest temperature:")
sensor_df.orderBy(desc("Temperature")).select("Machine","Location","Temperature").show(1)

print("Highest average vibration:")
sensor_df.groupBy("Machine").agg(avg("Vibration").alias("Avg_Vibration")).orderBy(desc("Avg_Vibration")).show(1)

print("Total warnings:")
print(health_df.filter(col("Machine_Status")=="WARNING").count())

print("Warnings by location:")
(health_df.filter(col("Machine_Status")=="WARNING")
 .groupBy("Location").count().orderBy(desc("count")).show())

print("Updated rule:")
updated_health=sensor_df.withColumn(
    "Status",when((col("Temperature")>92)|(col("Vibration")>8.5),"WARNING").otherwise("NORMAL"))
updated_health.groupBy("Status").count().show()


## 🧠 Final Takeaway
Hadoop → distributed storage/processing ecosystem. HDFS → distributed storage. MapReduce → distributed processing model. Spark → fast general-purpose distributed processing. Streaming → continuously arriving data with low latency.

**Big Data is ultimately about turning large-scale data into useful business value.**


In [ ]:
spark.stop()
print('Spark session stopped.')
